# TER — Fine-tuning RoBERTa sur SNLI avec Optuna (version GitHub propre)

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

import evaluate

## 1. Configuration générale

On fixe la graine aléatoire pour obtenir des résultats plus reproductibles. La recherche Optuna est faite sur un sous-échantillon de 55 000 exemples pour réduire le coût de calcul. L'entraînement final est ensuite réalisé sur tout le train SNLI.

In [ ]:
# ==============================
# Configuration générale
# ==============================

SEED = 42
set_seed(SEED)

MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
NUM_LABELS = 3

# Mettre RUN_OPTUNA = True pour relancer la recherche d'hyperparamètres.
# Par défaut, on conserve les meilleurs paramètres trouvés pendant le TER afin d'éviter de relancer une étape coûteuse.
RUN_OPTUNA = False
OPTUNA_TRAIN_SIZE = 55_000
OPTUNA_VALIDATION_SIZE = 5_000
N_TRIALS = 10

BEST_PARAMS = {
    "learning_rate": 2e-05,
    "per_device_train_batch_size": 32,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
}

# Chemins publics et réutilisables : aucune dépendance à un Drive personnel.
BASE_DIR = os.environ.get("TER_OUTPUT_DIR", ".")
OPTUNA_OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "optuna_roberta_snli")
FINAL_OUTPUT_DIR = os.path.join(BASE_DIR, "models", "final_roberta_snli_model")
LOGGING_DIR = os.path.join(BASE_DIR, "logs", "roberta_snli")
RESULTS_JSON_PATH = os.path.join(BASE_DIR, "results", "results_roberta_snli.json")
os.makedirs(OPTUNA_OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_JSON_PATH), exist_ok=True)

USE_FP16 = torch.cuda.is_available()

print("Modèle utilisé :", MODEL_NAME)
print("RUN_OPTUNA :", RUN_OPTUNA)
print("Dossier du modèle final :", FINAL_OUTPUT_DIR)
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

## 2. Chargement du tokenizer et du dataset SNLI

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Chargement du dataset SNLI...")
dataset = load_dataset("snli")

# On retire les exemples avec label = -1 car ils ne sont pas correctement annotés.
train_dataset_full = dataset["train"].filter(lambda x: x["label"] != -1)
validation_dataset_full = dataset["validation"].filter(lambda x: x["label"] != -1)
test_dataset_full = dataset["test"].filter(lambda x: x["label"] != -1)

print("Train complet :", len(train_dataset_full))
print("Validation complète :", len(validation_dataset_full))
print("Test complet :", len(test_dataset_full))
print("Labels SNLI :", dataset["train"].features["label"].names)

## 3. Création des datasets Optuna et des datasets finaux

- **Optuna** : 55 000 exemples d'entraînement + 5 000 exemples de validation.
- **Final** : tout le train SNLI + validation complète + test complet.

In [ ]:
# ==============================
# Préparation des datasets Optuna et finaux
# ==============================

train_dataset_shuffled = train_dataset_full.shuffle(seed=SEED)
validation_dataset_shuffled = validation_dataset_full.shuffle(seed=SEED)

# Sous-ensemble utilisé uniquement pour la recherche Optuna
optuna_train_dataset = train_dataset_shuffled.select(
    range(min(OPTUNA_TRAIN_SIZE, len(train_dataset_shuffled)))
)
optuna_validation_dataset = validation_dataset_shuffled.select(
    range(min(OPTUNA_VALIDATION_SIZE, len(validation_dataset_shuffled)))
)

# Dataset complet pour l'entraînement final
final_train_dataset = train_dataset_shuffled
final_validation_dataset = validation_dataset_shuffled
final_test_dataset = test_dataset_full

print("Train Optuna :", len(optuna_train_dataset))
print("Validation Optuna :", len(optuna_validation_dataset))
print("Train final complet :", len(final_train_dataset))
print("Validation finale complète :", len(final_validation_dataset))
print("Test final complet :", len(final_test_dataset))

## 4. Tokenisation RoBERTa

RoBERTa reçoit une paire de phrases : `premise` et `hypothesis`. On limite la longueur à 128 tokens pour garder un temps de calcul raisonnable.

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

columns_to_remove = ["premise", "hypothesis"]

print("Tokenisation des datasets Optuna...")
tokenized_optuna_train = optuna_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)
tokenized_optuna_validation = optuna_validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

print("Tokenisation des datasets finaux...")
tokenized_final_train = final_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)
tokenized_final_validation = final_validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)
tokenized_final_test = final_test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

for ds in [tokenized_optuna_train, tokenized_optuna_validation, tokenized_final_train, tokenized_final_validation, tokenized_final_test]:
    ds.set_format("torch")

print("Tokenisation terminée.")

## 5. Métriques d'évaluation

On suit la logique du TER : accuracy pour la performance globale et macro-F1 pour vérifier que les trois classes sont prises en compte de manière équilibrée.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )

    macro_f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro",
    )

    return {
        "accuracy": accuracy["accuracy"],
        "macro_f1": macro_f1["f1"],
    }

## Recherche d'hyperparamètres avec Optuna

Cette étape permet de garder une trace reproductible de la recherche d'hyperparamètres. Elle est désactivée par défaut pour éviter de relancer une phase longue ; mettre `RUN_OPTUNA = True` dans la configuration pour l'exécuter.

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
    )


def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", [2, 3]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }


def compute_objective(metrics):
    return metrics["eval_macro_f1"]

In [ ]:
if RUN_OPTUNA:
    training_args_optuna = TrainingArguments(
        output_dir=OPTUNA_OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        fp16=USE_FP16,

        # Valeurs par défaut remplacées par Optuna pendant la recherche
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,

        seed=SEED,
        data_seed=SEED,
    )

    trainer_optuna = Trainer(
        model_init=model_init,
        args=training_args_optuna,
        train_dataset=tokenized_optuna_train,
        eval_dataset=tokenized_optuna_validation,
        compute_metrics=compute_metrics,
    )

    best_run = trainer_optuna.hyperparameter_search(
        direction="maximize",
        backend="optuna",
        hp_space=optuna_hp_space,
        compute_objective=compute_objective,
        n_trials=N_TRIALS,
    )

    print("Meilleur essai Optuna :")
    print(best_run)
    BEST_PARAMS = best_run.hyperparameters
else:
    print("Recherche Optuna non relancée. Paramètres conservés :")
    print(BEST_PARAMS)

## 7. Nettoyage mémoire avant l'entraînement final

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Mémoire nettoyée. Prêt pour l'entraînement final RoBERTa.")

## 8. Entraînement final de RoBERTa sur tout le train SNLI

Ici, on repart d'un modèle `roberta-base` pré-entraîné neuf, puis on l'entraîne sur le **dataset complet** avec les meilleurs hyperparamètres trouvés par Optuna.

In [ ]:
final_training_args = TrainingArguments(
    output_dir=FINAL_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=False,

    learning_rate=3.667953850131021e-05,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.08481300155385081,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    seed=SEED,
    data_seed=SEED,
)

In [ ]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_final_train,
    eval_dataset=tokenized_final_validation,
    compute_metrics=compute_metrics,
)

train_result = final_trainer.train()
validation_results = final_trainer.evaluate(tokenized_final_validation)

print("Résultats validation finale RoBERTa :")
print(validation_results)

## 9. Évaluation finale sur le test set SNLI

In [ ]:
test_results = final_trainer.evaluate(tokenized_final_test)

print("Résultats test final RoBERTa :")
print(test_results)

## 10. Sauvegarde du modèle final

In [ ]:
final_trainer.save_model(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print(f"Modèle RoBERTa final sauvegardé dans : {FINAL_OUTPUT_DIR}")

## 11. Courbes d'apprentissage RoBERTa

In [ ]:
log_history = pd.DataFrame(final_trainer.state.log_history)
log_history

In [ ]:
# Courbe de training loss
train_loss_df = log_history.dropna(subset=["loss"])[["epoch", "loss"]]

plt.figure(figsize=(8, 5))
plt.plot(train_loss_df["epoch"], train_loss_df["loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("RoBERTa — Évolution de la training loss")
plt.grid(True)
plt.show()

In [ ]:
# Courbe de validation loss
validation_loss_df = log_history.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]]

plt.figure(figsize=(8, 5))
plt.plot(validation_loss_df["epoch"], validation_loss_df["eval_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("RoBERTa — Évolution de la validation loss")
plt.grid(True)
plt.show()

## 12. Sauvegarde des résultats pour comparaison avec BERT

Le fichier `results_roberta.json` pourra être chargé dans un notebook séparé de comparaison avec `results_bert.json`.

In [ ]:
best_params ={
    "learning_rate": 3.667953850131021e-05,
    "per_device_train_batch_size": 16,
    "num_train_epochs": 2,
    "weight_decay": 0.08481300155385081
}

In [ ]:
summary_results = {
    "model": "RoBERTa",
    "model_name": MODEL_NAME,
    "final_train_size": len(final_train_dataset),
    "final_validation_size": len(final_validation_dataset),
    "final_test_size": len(final_test_dataset),
    "best_hyperparameters": best_params,
    "validation_results": validation_results,
    "test_results": test_results,
}

with open(RESULTS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_results, f, indent=4)

results_table = pd.DataFrame([
    {
        "model": "RoBERTa",
        "validation_accuracy": validation_results.get("eval_accuracy"),
        "validation_macro_f1": validation_results.get("eval_macro_f1"),
        "validation_loss": validation_results.get("eval_loss"),
        "test_accuracy": test_results.get("eval_accuracy"),
        "test_macro_f1": test_results.get("eval_macro_f1"),
        "test_loss": test_results.get("eval_loss"),
        "learning_rate": best_params.get("learning_rate"),
        "batch_size": best_params.get("per_device_train_batch_size"),
        "epochs": best_params.get("num_train_epochs"),
        "weight_decay": best_params.get("weight_decay"),
    }
])

results_table.to_csv(RESULTS_CSV_PATH, index=False)

print(f"Résultats JSON sauvegardés dans : {RESULTS_JSON_PATH}")
print(f"Résultats CSV sauvegardés dans : {RESULTS_CSV_PATH}")
results_table

In [ ]:
# Option Colab : décommentez si vous souhaitez monter Google Drive.
final_trainer.save_model(
    "BASE_DIR/final_roberta_snli_model"
)

tokenizer.save_pretrained(
    "BASE_DIR/final_roberta_snli_model"
)